# 02 · Limpieza tipada y contratos de identificadores

**Pregunta.** ¿Es posible producir una capa limpia, reproducible y sin pérdidas, dejando explícito qué identificadores son sólo observados y qué armonización territorial sigue pendiente?

**Entradas.** Los 192 CSV auditados, `configs/harmonization.yaml`, el resumen crudo y la evidencia cartográfica registrada.

**Método.** Validación exacta de año/fecha, tipado explícito, serialización canónica de IDs, conservación de categorías, indicadores con denominador documentado y conciliación después de agregar. Cada año se escribe en Parquet de forma atómica.

**Política de retención.** No se excluye ninguna fila. Edades mayores de 100, distritos vacíos y denominadores identificados en cero se conservan con bandera o nulo semántico. `SEXO=2` permanece como categoría fuente de 2024.

**Límite.** `section_key_observed` evita colisiones dentro de una elección, pero no demuestra equivalencia geométrica entre elecciones. El agregado sección–año de este notebook es provisional y no es un panel longitudinal armonizado.

**Salidas.** Parquet limpios por año, resumen, bitácora de retención, agregado provisional, conciliación y transiciones nominales de códigos.

## Entorno y precondiciones

Se detecta la raíz y se comprueba que la auditoría previa existe. El notebook no depende del estado de memoria de los notebooks 00 o 01.

In [ ]:
from pathlib import Path
import resource
import time

import pandas as pd
import yaml
from IPython.display import Markdown, display


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "configs/harmonization.yaml").exists():
            return candidate
    raise FileNotFoundError("No se encontró la raíz del proyecto")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
RUN_STARTED = time.perf_counter()
required_inputs = [
    PROJECT_ROOT / "artifacts/tables/raw_year_summary.json",
    PROJECT_ROOT / "artifacts/tables/raw_section_frame_inventory.parquet",
    PROJECT_ROOT / "configs/harmonization.yaml",
    PROJECT_ROOT / "configs/geography_sources.yaml",
]
input_checks = {path.relative_to(PROJECT_ROOT).as_posix(): path.exists() for path in required_inputs}
display(pd.Series(input_checks, name="existe").to_frame())
assert all(input_checks.values()), "Faltan entradas auditadas del notebook 01"


In [ ]:
display(Markdown(
    f"**Interpretación.** Las `{len(required_inputs)}` precondiciones existen. "
    "La transformación parte de artefactos persistidos y configuraciones versionadas."
))

## Contrato temporal, categórico y de IDs

Las fechas se mapean por valor fuente exacto porque 2009–2021 usan día/mes/año y 2024 usa mes/día/año. Los anchos son reglas de serialización, no afirmaciones sobre una clave maestra oficial.

In [ ]:
with (PROJECT_ROOT / "configs/harmonization.yaml").open(encoding="utf-8") as stream:
    harmonization_config = yaml.safe_load(stream)

year_contract = pd.DataFrame([
    {"year": int(year), **values}
    for year, values in harmonization_config["years"].items()
])
display(year_contract)
display(pd.Series(harmonization_config["id_serialization"], name="regla").to_frame())
display(pd.Series(harmonization_config["retention_policy"], name="regla").to_frame())

In [ ]:
display(Markdown(
    "**Interpretación.** El contrato fija seis fechas inequívocas y conserva ceros a la izquierda. "
    "La llave `section_key_observed = entidad:sección` se limita al marco declarado por cada fuente; "
    "no recibe el nombre de ID armonizado."
))

## Evidencia y estado de la cartografía

Se muestra el registro de fuentes cartográficas. La ausencia de descarga oficial directa se trata como un bloqueo verificable, no como permiso para inferir polígonos por coincidencia de códigos.

In [ ]:
with (PROJECT_ROOT / "configs/geography_sources.yaml").open(encoding="utf-8") as stream:
    geography_config = yaml.safe_load(stream)

geography_sources = pd.DataFrame(geography_config["sources"])
display(geography_sources)
display(pd.Series({
    "status": geography_config["status"],
    "crosswalk_downloaded": geography_config["crosswalk_downloaded"],
    "geometry_downloaded": geography_config["geometry_downloaded"],
    "decision": geography_config["decision"],
}, name="valor").to_frame())

In [ ]:
display(Markdown(
    "**Hallazgo.** El catálogo especializado requiere solicitud de información. El informe DERFE "
    "registra 69,548 secciones para mayo de 2023, mientras la fuente CCPC 2024 contiene 70,751 "
    "códigos observados. La discrepancia impide declarar que la base 2024 ya está en un marco "
    "seccional 2023 único y validado."
))

## Transformación limpia por año

La función versionada lee cada columna como texto, valida año y literal de fecha, convierte tipos y escribe seis archivos Parquet. Después vuelve a leer cada salida para validar unicidad, formatos, categorías, no negatividad y la identidad `LN = SV + NV + NS`.

In [ ]:
from radar27.clean.harmonize import harmonize_all

clean_started = time.perf_counter()
summary, exclusions, reconciliation, transitions = harmonize_all(PROJECT_ROOT)
clean_seconds = time.perf_counter() - clean_started
peak_rss_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024

display(summary[[
    "year", "election_family", "election_date", "raw_rows", "clean_rows", "excluded_rows",
    "unique_record_ids", "accounting_mismatches", "age_review_rows",
    "missing_federal_district_rows", "missing_local_district_rows", "zero_identified_rows",
    "non_binary_rows",
]])
display(pd.DataFrame({
    "elapsed_seconds": [round(clean_seconds, 2)],
    "peak_rss_mb_process": [round(peak_rss_mb, 1)],
    "clean_rows": [int(summary.clean_rows.sum())],
    "excluded_rows": [int(summary.excluded_rows.sum())],
}))

In [ ]:
display(Markdown(
    f"**Resultado.** Se tiparon y validaron `{int(summary.clean_rows.sum()):,}` filas en "
    f"`{clean_seconds:.1f}` s, con `{int(summary.excluded_rows.sum()):,}` exclusiones. "
    "Cada `record_id` es único y todos los totales concilian con la auditoría cruda."
))

## Retención, banderas y nulos semánticos

La tabla siguiente distingue exclusión de revisión. Las banderas pueden traslaparse; sus conteos no se suman para obtener filas únicas.

In [ ]:
display(exclusions)
assert exclusions["excluded_rows"].sum() == 0
assert summary["excluded_rows"].sum() == 0

In [ ]:
display(Markdown(
    "**Interpretación.** Ninguna observación fue descartada. Los distritos vacíos permanecen como "
    "nulos, las edades mayores a 100 quedan marcadas para revisión y los cocientes con denominador "
    "cero se conservan como nulos en vez de convertirse artificialmente en cero."
))

## Conciliación del agregado provisional

Los conteos se suman primero por elección y código observado de sección; los porcentajes se calculan después como razón de sumas. El nombre y estado del artefacto impiden confundirlo con un panel armonizado.

In [ ]:
display(reconciliation)
raw_summary = pd.read_json(PROJECT_ROOT / "artifacts/tables/raw_year_summary.json")
comparison = reconciliation.merge(
    raw_summary[["year", "rows", "sum_ln", "sum_sv", "sum_nv", "sum_ns"]],
    left_on="election_year", right_on="year", validate="one_to_one",
)
comparison["rows_match"] = comparison["demographic_rows"].eq(comparison["rows"])
comparison["ln_match"] = comparison["sum_nominal_list"].eq(comparison["sum_ln"])
comparison["sv_match"] = comparison["sum_voted"].eq(comparison["sum_sv"])
comparison["nv_match"] = comparison["sum_did_not_vote"].eq(comparison["sum_nv"])
comparison["ns_match"] = comparison["sum_unspecified"].eq(comparison["sum_ns"])
display(comparison[["election_year", "rows_match", "ln_match", "sv_match", "nv_match", "ns_match"]])
assert comparison[["rows_match", "ln_match", "sv_match", "nv_match", "ns_match"]].all().all()

In [ ]:
display(Markdown(
    f"**Resultado.** Las `{len(reconciliation)}` elecciones conservan exactamente filas y cuatro "
    "totales contables después de agregar. Esto valida la transformación aritmética, no la "
    "equivalencia territorial entre años."
))

## Transiciones nominales y bloqueo longitudinal

Se comparan altas, bajas y cambios de metadatos entre elecciones consecutivas. Coincidencia nominal significa sólo que se repite `entidad:sección`.

In [ ]:
display(transitions)

In [ ]:
latest = transitions.iloc[-1]
display(Markdown(
    f"**Hallazgo.** En 2021–2024 hay `{int(latest.shared_observed_codes):,}` códigos nominales "
    f"compartidos, `{int(latest.only_previous):,}` sólo previos y `{int(latest.only_current):,}` "
    "sólo actuales, además de cambios de metadatos. No se genera una llave longitudinal."
))

## Cierre y compuertas

Se comprueba la capa limpia y se mantiene fallida, de forma deliberada, la compuerta que requeriría cartografía/crosswalk oficial.

In [ ]:
required_outputs = [
    PROJECT_ROOT / "artifacts/tables/harmonization_year_summary.csv",
    PROJECT_ROOT / "artifacts/tables/cleaning_exclusion_log.csv",
    PROJECT_ROOT / "artifacts/tables/provisional_section_year_totals.parquet",
    PROJECT_ROOT / "artifacts/tables/aggregation_reconciliation.csv",
    PROJECT_ROOT / "artifacts/tables/section_code_transition_summary.csv",
]
checks = {
    "all_54m_rows_retained": int(summary.clean_rows.sum()) == 54_872_712,
    "zero_excluded_rows": int(summary.excluded_rows.sum()) == 0,
    "record_ids_unique": bool((summary.clean_rows == summary.unique_record_ids).all()),
    "accounting_holds": bool(summary.accounting_mismatches.eq(0).all()),
    "all_outputs_exist": all(path.exists() for path in required_outputs),
    "official_crosswalk_available": bool(geography_config["crosswalk_downloaded"]),
    "longitudinal_panel_built": False,
}
display(pd.Series(checks, name="aprobado").to_frame())
elapsed = time.perf_counter() - RUN_STARTED
display(Markdown(
    f"**Cierre.** Pasaron `{sum(checks.values())}/{len(checks)}` comprobaciones en `{elapsed:.1f}` s. "
    "Las dos negativas son restricciones metodológicas intencionales: no existe crosswalk oficial "
    "validado y, por ello, no se construyó el panel longitudinal. Siguiente paso: solicitar los "
    "productos cartográficos especializados y validar una estrategia de correspondencia."
))